# DALI 蛋白质比对工作流

这个 notebook 面向你服务器上的 JupyterLab，帮助你快速用 DALI 作结构比对并保存结果。

## 目标

1. 用本地或远程 DALI 安装对查询结构进行比对。
2. 以批处理方式提交多个 PDB。
3. 把结果转换成可视化或下游分析需要的格式。

## 前提条件

- Python 3.10+（服务器已安装）。
- DALI 路径（例如 `dali.pl` 脚本或 `dali.sh`）在 `PATH` 中或设置为绝对路径。
- 如果调用远程 DALI API，请确保网络允许访问 `ekhidna2.biocenter.helsinki.fi`。
- 结构文件统一放在 `data/structures/` 下，并使用 `.pdb` 或 `.cif`。


## DALI 数据库准备
- 本地 DALI 安装默认依赖 `pdb100` 或 `pdb90` 数据库，请确保磁盘有 ≥50 GB 空间。
- 推荐做法是在服务器上运行 `setup_dali_db.sh`（或参考官方 `fetch_dali.pl`）定期拉取最新 PDB 库。
- 若仅做远程比对，可把 `DALI_DB_DIR` 环境变量指向挂载盘，然后在命令行参数中加入 `-databank <path>`。
- 完成下载后更新 `PATH` 或 `DALI_CMD`，并把数据库路径加入 `~/.bashrc` 方便 Notebook 直接调用。

## 工作流程概览

1. 初始化路径与依赖。
2. 枚举 query 结构文件。
3. 对每个结构运行本地 DALI 并捕获日志。
4. 解析输出，导出排名，必要时可视化。

In [ ]:
import logging
import shutil
import subprocess
from pathlib import Path
from typing import Iterable

import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

In [ ]:
ROOT = Path().resolve()
STRUCTURES_DIR = ROOT / "data" / "structures"
OUTPUT_BASE = ROOT / "outputs" / "dali"
ESM3_PRED_DIR = ROOT / "outputs" / "esm3" / "predictions"

DALI_CMD = Path("dali.pl")

if not STRUCTURES_DIR.exists():
    raise FileNotFoundError(f"{STRUCTURES_DIR} 不存在，请先准备结构文件。")

if not DALI_CMD.exists():
    dali_in_path = shutil.which("dali.pl")
    if dali_in_path:
        DALI_CMD = Path(dali_in_path)
    else:
        logging.warning("未找到 dali.pl，运行前请设置 DALI_CMD 为绝对路径。")

OUTPUT_BASE.mkdir(parents=True, exist_ok=True)


## 查询结构准备

列出所有待比对的 PDB/CIF/ENT 文件，确保命名一致，便于批处理。

## ESM3 结果整合（萜合酶探索）
为了锁定潜在萜类合酶，可以把 ESM3 Workflow 的预测结构（PDB）自动同步到 `data/structures/`，然后交给 DALI 做结构相似性搜索。
- 默认假设预测结果保存在 `outputs/esm3/predictions/`，文件名和 `target_id` 一致。
- 同步时会为每个文件加上 `esm3_` 前缀，避免覆盖手工准备的结构。
- 结合 Z-score、高度保守的活性位点和注释，可以快速筛查疑似萜合酶。

In [ ]:
def sync_esm3_predictions(source_dir: Path = ESM3_PRED_DIR, target_dir: Path = STRUCTURES_DIR) -> list[Path]:
    synced = []
    if not source_dir.exists():
        logging.info("ESM3 预测目录 %s 不存在，跳过同步。", source_dir)
        return synced
    target_dir.mkdir(parents=True, exist_ok=True)
    for ext in ("*.pdb", "*.cif"):
        for structure in sorted(source_dir.glob(ext)):
            dest = target_dir / f"esm3_{structure.name}"
            if not dest.exists() or structure.stat().st_mtime > dest.stat().st_mtime:
                shutil.copy2(structure, dest)
                logging.info("同步 %s -> %s", structure.name, dest.name)
            synced.append(dest)
    return synced

In [ ]:
synced_esm3 = sync_esm3_predictions()
query_patterns = ["*.pdb", "*.cif", "*.ent"]
queries: list[Path] = []
for pattern in query_patterns:
    queries.extend(sorted(STRUCTURES_DIR.glob(pattern)))
if synced_esm3:
    logging.info("已加入 %d 个 ESM3 预测结构用于 DALI。", len(synced_esm3))
# 去重保持顺序
unique: list[Path] = []
seen: set[Path] = set()
for path in queries:
    if path not in seen:
        unique.append(path)
        seen.add(path)
queries = unique
if not queries:
    logging.warning("在 %s 中没有发现 PDB/CIF 文件，请先放入结构。", STRUCTURES_DIR)
else:
    print(
        "找到 %d 个结构文件，前 5 个：" % len(queries),
        *[q.name for q in queries[:5]],
        sep="\n",
    )

## DALI 结构比对

提供一个可复用的函数来安全运行 `dali.pl`，并可批量处理多个 query。

In [ ]:
def run_local_dali(query_path: Path, output_dir: Path) -> Path:
    output_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        str(DALI_CMD),
        "-query", str(query_path),
        "-hera", str(output_dir),
    ]
    logging.info("运行 DALI：%s -> %s", query_path.name, output_dir)
    process = subprocess.run(cmd, capture_output=True, text=True)
    if process.returncode != 0:
        raise RuntimeError(
            f"DALI 执行失败（{query_path.name}）：{process.stderr.strip()}"
        )
    logging.info("完成 %s，日志写入 %s", query_path.name, output_dir / "dali.log")
    return output_dir / "dali.log"

def run_batch(queries: Iterable[Path]) -> list[Path]:
    logs = []
    for query in queries:
        target_dir = OUTPUT_BASE / query.stem
        log_path = run_local_dali(query, target_dir)
        logs.append(log_path)
    return logs

## 解析 DALI 结果

把 `dali.log` 解析成表格便于排序和导出。

In [ ]:
def parse_dali_log(log_path: Path) -> pd.DataFrame:
    sightings = []
    with log_path.open(encoding="utf-8") as fh:
        for line in fh:
            if line.startswith('#') or not line.strip():
                continue
            parts = line.split()
            if len(parts) < 9:
                continue
            sightings.append({
                "rank": int(parts[0]),
                "pdb": parts[1],
                "z_score": float(parts[5]),
                "rmsd": float(parts[7]),
            })
    return pd.DataFrame(sightings)

def summarize_logs(log_paths: Iterable[Path]) -> pd.DataFrame:
    records = []
    for log_path in log_paths:
        if not log_path.exists():
            continue
        df = parse_dali_log(log_path)
        if df.empty:
            continue
        df = df.assign(target=log_path.parent.name)
        records.append(df)
    if not records:
        return pd.DataFrame()
    return pd.concat(records, ignore_index=True)


In [ ]:
log_files = sorted(OUTPUT_BASE.rglob("dali.log"))
summary = summarize_logs(log_files)
if summary.empty:
    print("当前尚无 DALI 结果，先运行一次比对。")
else:
    summary = summary.sort_values(by="z_score", ascending=False)
    display(summary.head(10))
    summary_path = OUTPUT_BASE / "dali_summary.csv"
    summary.to_csv(summary_path, index=False)
    logging.info("摘要已经保存到 %s", summary_path)


## 示例执行

若已经准备好了结构文件，可先跑一条记录“绿灯”全链路。

In [ ]:
if queries:
    sample_query = queries[0]
    target = OUTPUT_BASE / sample_query.stem
    log_path = run_local_dali(sample_query, target)
    print("示例日志：", log_path)
else:
    print("未找到结构文件，请先放入 data/structures。")


## 下一步

- 如果需要可视化，可将 `summary.head(N)` 传给 `nglview`/`py3Dmol`。
- 可将 `run_batch(queries)` 包装进 `papermill`/`nbconvert` 生成报告。
- 若要上传到远程 DALI，请在这个 notebook 中替换 `subprocess` 调用为 API 请求。